# 29. 分组、聚合与数据透视

<!-- module-learning-arc:start -->
> **Pandas 模块主线｜第 8 / 10 步：从明细得到分组指标**
>
> **持续应用背景：** 搭建电商履约异常追踪台：把订单、客户、商品和履约信息整理成安全合并的事实表，再生成趋势指标和异常工单。
>
> **承接上一阶段：** 数据读取与保存  →  **本章任务：** 分组、聚合与数据透视  →  **下一步：** 数据合并与结构转换
>
> **大作业连接：** 本章练习将成为《电商履约异常追踪台》的一部分，最终需要从多表质量审计走到订单粒度事实表、窗口趋势和可复核异常工单。
<!-- module-learning-arc:end -->


## 本章场景

处理一张订单或销售表时，几乎都会冒出同一类问题——“按地区算一下总额”“按部门数一数人数”。



## 本章目标

学完本章，你将能够：

- **理解**：理解 groupby、聚合函数与 pivot_table 的分组/透视逻辑。
- **操作**：能按列分组并聚合，做透视与交叉表。
- **迁移**：能按地区/渠道/品类分组汇总出经营报表并解读。


## 29.1 核心概念

**背景引入**：处理一张订单或销售表时，几乎都会冒出同一类问题——“按地区算一下总额”“按部门数一数人数”。手动一行行去加显然不现实，而分组、聚合与透视正是为这类“按维度汇总”量身定做的招式：一条命令就能把几百行数据压成一张自己能看懂的汇总表。这一章你学会后，日常看数、写周报、搭报表都会顺手很多。

- 分组前必须明确维度、指标和聚合函数。
- agg压缩行数，transform保持原行数。
- 透视表中的缺失组合与真实0含义不同。

> **直观类比**：分组就像把全班按小组分开：agg 只交回“每组一个数字”（表变短）；transform 却把每组结果“广播”回每个组员（行数不变）。正是这样，才能给每笔订单标上“它所在地区的总额”，再逐行算占比。


## 29.2 方法分类速查

先用这张表建立本章的方法地图；每一行后面都有对应的独立示例或练习。

| 类别 | 常用方法或写法 | 主要用途 | 需要特别注意 |
| --- | --- | --- | --- |
| groupby() | `pd.DataFrame()`、`orders.groupby()`、`.sum()`、`sum()` | 分组前先明确分组维度和要计算的指标。 | 使用mean却把结果描述为合计 |
| agg() | `pd.DataFrame()`、`orders.groupby()`、`.agg()`、`.rename()` | agg可以一次计算多个统计量；这里使用旧版Pandas也支持的写法。 | groupby后忘记处理索引 |
| transform() | `pd.DataFrame()`、`orders.groupby()`、`.transform()`、`orders['region_total']` | transform结果和原表等长，可以直接生成派生列。 | 把缺失组合无条件填0 |
| pivot_table() | `pd.DataFrame()`、`orders.pivot_table()` | index、columns和values分别定义透视表的行、列和指标。 | 使用mean却把结果描述为合计 |
| crosstab() | `pd.DataFrame()`、`pd.crosstab()`、`orders['region']`、`orders['channel']` | crosstab适合快速统计两个分类变量的组合频数。 | groupby后忘记处理索引 |


## 29.3 示例 1：分组聚合

命名聚合让输出列直接表达口径。

**背景引入**：订单表一行行堆着订单，老板一句“每个地区卖了多少、一共几单、平均客单价多少”，总不能拿计算器挨个加。把订单按“地区”分成组，一次算出合计、单数、均值，一张汇总表回答三个问题。

**讲解**：用 groupby 按 region 分组，再用 agg 命名聚合一次算多个指标，输出列名直接表达口径。

- groupby 前先想清楚“按哪列分组、算什么指标”，别拿 mean 的结果说成合计；
- agg 的写法是 `新列名 = ("原列", "函数")`，一行写多个指标，结果自动成为新列名；
- `size` 数的是行数（订单数），配合 `sum`（金额合计）、`mean`（平均金额）一次看清地区表现；
- **口诀**：先分组后聚合，列名即口径，sum 是合计、size 数单数、mean 算平均。


<!-- math-foundation:chapter-29 -->
### 数学推导｜分组聚合与加权平均

> 阅读方法：先跟着步骤理解每个量怎样产生，再看最后的可计算形式；不需要脱离业务场景死记公式。

**第 1 步｜组均值来自组内总和与组内数量。** 记 $S_g=\sum_{i:c_i=g}x_i$，则 $\bar{x}_g=S_g/n_g$。

**第 2 步｜合并组均值时恢复各组权重。** 总体均值不是组均值的简单平均，而是

$$
\bar{x}=\frac{\sum_g n_g\bar{x}_g}{\sum_g n_g}
$$

**第 3 步｜推广到一般权重。** 把样本量 $n_g$ 换成任意非负权重 $w_i$，就得到加权平均 $\sum_iw_ix_i/\sum_iw_i$。

**把上面的关系收束为本章计算式：**

$$
\bar{x}_g=\frac{1}{n_g}\sum_{i:c_i=g}x_i,\qquad \bar{x}_w=\frac{\sum_iw_ix_i}{\sum_iw_i}
$$

**符号解释：** $g$ 是分组，$n_g$ 是组内样本量，$w_i$ 是权重。

**代码对应：** `groupby(...).agg(...)` 对应组内统计；加权平均需要显式计算分子和分母。

**使用边界：** 组均值不能在忽略样本量的情况下再次简单平均，否则会产生聚合偏差。


In [ ]:
import pandas as pd

orders = pd.DataFrame(
    {
        "region": ["华东", "华东", "华南", "华南", "华北", "华北"],
        "channel": ["线上", "线下", "线上", "线下", "线上", "线下"],
        "amount": [520, 310, 460, 280, 390, 260],
        "quantity": [3, 2, 2, 1, 2, 1],
    }
)
summary = orders.groupby("region").agg(
    sales=("amount", "sum"),
    orders=("amount", "size"),
    average=("amount", "mean"),
)
print(summary)


## 29.4 示例 2：transform组内占比

transform结果与原表等长，可直接添加为新列。

**背景引入**：想算“每个地区里，各渠道订单占本地区总额的百分之几”。直接 groupby().sum() 汇总后只剩几行汇总结果，和原表对不上号——transform 把每组合计“广播”回每一行，占比就能逐行算出来。

**讲解**：transform 返回和原表等长的一列，添加到原表后逐行相除就是组内占比。

- `groupby("region")["amount"].transform("sum")` 给每行补上它所在地区的金额合计，行数不变；
- 用 `amount / region_total` 逐行相除，得到每个订单在本地区的占比；
- 遇到缺失组合先查清原因，不要无条件填 0，否则占比会失真；
- **口诀**：汇总要回到每一行就用 transform，除以组内合计即得占比。


In [ ]:
orders["region_total"] = orders.groupby("region")["amount"].transform("sum")
orders["region_share"] = orders["amount"] / orders["region_total"]
print(orders)


## 29.5 示例 3：透视表与交叉表

index与columns分别定义行维度和列维度。

**背景引入**：老板要看“地区 × 渠道”的二维销售矩阵——华东线上多少、华南线下多少，一屏看完。手动分组再拼接太绕，透视表把行维度、列维度、指标一次摆好，和 Excel 数据透视表一个思路。

**讲解**：pivot_table 用 index 定行、columns 定列、values 定指标，aggfunc 决定怎么汇总；crosstab 专门统计两个分类变量的组合频数。

- `pivot_table` 的三个关键参数别混：index 是行（region）、columns 是列（channel）、values 是要汇总的数值（amount），aggfunc="sum" 用合计；
- `fill_value=0` 把没有出现的组合补成 0，透视表读起来更干净；
- `pd.crosstab` 数的是“组合出现了多少次”，`margins=True` 会额外加合计行和列；
- **口诀**：index 行、columns 列、values 数值，crosstab 管频数、pivot_table 管汇总。


In [ ]:
pivot = orders.pivot_table(
    index="region",
    columns="channel",
    values="amount",
    aggfunc="sum",
    fill_value=0,
)
counts = pd.crosstab(orders["region"], orders["channel"], margins=True)
print(pivot)
print(counts)


## 29.6 核心操作独立示例

下面每个代码单元格只演示一个核心方法、函数或语法操作。请先阅读方法名称和任务说明，再单独运行当前单元格；示例尽量自带最小输入，不要求依赖前一个单元格留下的变量。


In [ ]:
# groupby()
# 分组前先明确分组维度和要计算的指标。
import pandas as pd

orders = pd.DataFrame(
    {"region": ["华东", "华东", "华南"], "amount": [320, 880, 460]}
)
print(orders.groupby("region")["amount"].sum())


In [ ]:
# agg()
# agg可以一次计算多个统计量；这里使用旧版Pandas也支持的写法。
import pandas as pd

orders = pd.DataFrame(
    {"region": ["华东", "华东", "华南"], "amount": [320, 880, 460]}
)
summary = (
    orders.groupby("region")["amount"]
    .agg(["sum", "mean"])
    .rename(columns={"sum": "total", "mean": "average"})
)
print(summary)


In [ ]:
# transform()
# transform结果和原表等长，可以直接生成派生列。
import pandas as pd

orders = pd.DataFrame(
    {"region": ["华东", "华东", "华南"], "amount": [320, 880, 460]}
)
orders["region_total"] = orders.groupby("region")["amount"].transform("sum")
orders["share"] = orders["amount"] / orders["region_total"]
print(orders)


In [ ]:
# pivot_table()
# index、columns和values分别定义透视表的行、列和指标。
import pandas as pd

orders = pd.DataFrame(
    {
        "region": ["华东", "华东", "华南"],
        "channel": ["线上", "线下", "线上"],
        "amount": [320, 880, 460],
    }
)
print(
    orders.pivot_table(
        index="region",
        columns="channel",
        values="amount",
        aggfunc="sum",
        fill_value=0,
    )
)


In [ ]:
# crosstab()
# crosstab适合快速统计两个分类变量的组合频数。
import pandas as pd

orders = pd.DataFrame(
    {"region": ["华东", "华东", "华南"], "channel": ["线上", "线下", "线上"]}
)
print(pd.crosstab(orders["region"], orders["channel"]))


**练一练 25.6**：沿用分组聚合的数据创建订单表 orders = pd.DataFrame({"region": ["华东", "华东", "华南"], "amount": [320, 880, 460]})，完成三件事：按地区分组求和并打印；用 transform 生成每组总额列 region_total 并打印组内占比 share；再补一列渠道 channel = ["线上", "线下", "线上"]，用 pivot_table 构造一张“地区 × 渠道”的透视表并打印。


In [ ]:
# 请在下方填写代码
import pandas as pd

# 任务1：按地区分组求和（groupby + sum）
# 任务2：用 transform 生成每组总额列，再算组内占比
# 任务3：补一列渠道，构造“地区 × 渠道”透视表
# TODO：请在下方完成 —— 练一练 25.6：沿用分组聚合的数据创建订单表 orders = pd.DataFrame({"region": ["华


In [ ]:
import pandas as pd

orders = pd.DataFrame(
    {"region": ["华东", "华东", "华南"], "amount": [320, 880, 460]}
)

# 任务1：按地区分组求和
total_by_region = orders.groupby("region")["amount"].sum()
print(total_by_region)

# 任务2：用 transform 生成每组总额列，再算组内占比
orders["region_total"] = orders.groupby("region")["amount"].transform("sum")
orders["share"] = orders["amount"] / orders["region_total"]
print(orders)

# 任务3：补一列渠道，构造“地区 × 渠道”透视表
orders["channel"] = ["线上", "线下", "线上"]
pivot = orders.pivot_table(
    index="region",
    columns="channel",
    values="amount",
    aggfunc="sum",
    fill_value=0,
)
print(pivot)


## 29.7 公开大型数据实战

下面使用 UCI Machine Learning Repository 的 Online Retail 公开数据集。原始数据包含 541,909 条英国在线零售交易，本课程使用固定随机种子抽取的 200,000 行子集。分析时在完整子集上计算，只展示摘要或少量样本。


In [ ]:
import numpy as np
import pandas as pd

# UCI Machine Learning Repository: Online Retail
# 原始数据 541,909 行；课程使用固定随机种子抽取的 200,000 行子集。
data_url = "/datasets/uci_online_retail_200k.csv"
large_orders = pd.read_csv(
    data_url,
    parse_dates=["InvoiceDate"],
    dtype={
        "InvoiceNo": "string",
        "StockCode": "string",
        "Description": "string",
        "Country": "category",
    },
).rename(
    columns={
        "InvoiceNo": "order_id",
        "StockCode": "stock_code",
        "Description": "description",
        "Quantity": "quantity",
        "InvoiceDate": "order_time",
        "UnitPrice": "unit_price",
        "CustomerID": "customer_id",
        "Country": "country",
    }
)
large_orders["sales"] = (
    large_orders["quantity"] * large_orders["unit_price"]
).round(2)
large_orders["status"] = np.where(
    large_orders["order_id"].str.startswith("C")
    | (large_orders["quantity"] < 0),
    "取消/退货",
    "完成",
)
print("UCI Online Retail 公开数据：")
print(f"  {len(large_orders):,} 行 × {large_orders.shape[1]} 列")
print(
    "内存占用：", f"{large_orders.memory_usage(deep=True).sum() / 1024**2:.1f} MB"
)
large_orders.head()


In [ ]:
summary = (
    large_orders.query("status == '完成'")
    .groupby(
        [large_orders["order_time"].dt.to_period("M"), "country"],
        observed=True,
    )
    .agg(销售额=("sales", "sum"), 订单数=("order_id", "size"), 客单价=("sales", "mean"))
    .reset_index()
)
pivot = summary.pivot(index="order_time", columns="country", values="销售额")
print(f"聚合前 {len(large_orders):,} 行，聚合后 {len(summary):,} 行")
display(summary.head(10))
display(pivot.tail().round(0))


## 29.8 独立迁移练习

替换一个字段或分组口径，并核对处理前后的行数与粒度。

先在下面单元格完成自己的版本；需要参考时再回看紧邻的示例或参考实现。


In [ ]:
# TODO: 在此粘贴或改写最接近的示例。
# 记录：我改了什么？预期会发生什么？实际观察到什么？
change_note = "待填写"
expected_change = "待填写"
observed_change = "运行后填写"
print({"修改": change_note, "预期": expected_change, "观察": observed_change})


## 29.9 本章实训：分组汇总与粒度

这一组实验专门训练“观察一个结果 → 只改一个变量 → 解释变化”。先运行第一个代码单元格，再运行第二个。


In [ ]:
import pandas as pd

orders = pd.DataFrame(
    {
        "region": ["华东", "华东", "华南", "华南"],
        "channel": ["线上", "线下", "线上", "线下"],
        "sales": [120, 80, 150, 100],
    }
)
summary = orders.groupby("region", as_index=False)["sales"].sum()
print(summary)
print("汇总表每一行代表一个地区")


### 29.9.1 第一个结果怎么读

先确认明细表一行代表一笔订单，再确认汇总表一行代表一个地区。`groupby` 的字段决定结果的粒度。

请记录：输入是什么、输出是什么、输出支持了哪一个结论。



In [ ]:
orders["sales_level"] = orders["sales"].map(
    lambda value: "高" if value >= 120 else "普通"
)
print(orders)
print(orders["sales_level"].value_counts())


### 29.9.2 第二个结果怎么读

第二个实验只增加一个分类列，不改变原始销售额。练习解释：什么时候应该新增列，什么时候应该直接筛选行？

迁移任务：把一个输入值、一个字段或一个图表参数换成自己的例子，再用一句话解释变化。



## 29.10 错误恢复：脏数据转换怎么办

真实数据和真实代码都会出问题。本节先观察问题，再用一个明确的检查或修复步骤恢复运行。


In [ ]:
import pandas as pd

raw = pd.Series(["12", "unknown", "18", ""])
converted = pd.to_numeric(raw, errors="coerce")
print("转换结果：")
print(converted)
print("无法转换的数量：", converted.isna().sum())
print("后续可以选择删除、填充或回查原始值。")


### 29.10.1 错误恢复步骤

1. 先看错误类型、字段或数据形状。
2. 判断问题发生在输入、处理中间结果还是输出。
3. 修复后重新检查结果，而不是只让代码不报错。

errors="coerce" 会把无法转换的值记录为缺失，适合先完成质量盘点；不要在没有统计数量前直接删除。

迁移任务：把示例中的输入换成一组会触发问题的数据，并记录你的修复规则。



## 29.11 易错点提醒

- 使用mean却把结果描述为合计
- groupby后忘记处理索引
- 把缺失组合无条件填0


## 29.12 练习与作业

1. 按地区和渠道分组
2. 计算销售额、订单数和平均订单金额
3. 构建地区×渠道透视表

提交前检查：代码可从上到下运行，关键中间结果可核对，结论注明计算口径。

## 29.13 练习路径

1. **跟练**：先运行示例，确认输出结构，再完成“按地区和渠道分组”。
2. **独立完成**：不复制示例代码，完成“计算销售额、订单数和平均订单金额”，并保留一个中间结果用于检查。
3. **迁移挑战**：尝试“构建地区×渠道透视表”，用一两句话说明你修改了什么。

### 29.13.1 完成标准

- 代码从上到下运行不报错，关键变量类型和形状符合预期。
- 至少输出一个可核对的数值、表格或图形，并写明计算口径。
- 结论能够回答任务问题，同时说明一个限制或未验证的假设。

### 29.13.2 分级提示

- **提示 1**：先复用示例中的数据结构和变量命名。
- **提示 2**：把任务拆成“准备数据 → 计算 → 检查 → 表达”四步。
- **提示 3**：运行隐藏答案前，先用 type()、shape、head() 或断言定位问题。


In [ ]:
import pandas as pd

# TODO: 按地区和渠道分组，计算销售额、订单数和平均订单金额
# TODO: 构建地区×渠道透视表（销售额）
# TODO：请在下方完成 —— 25.13 练习与作业 1. 按地区和渠道分组 2. 计算销售额、订单数和平均订单金额 3. 构建地区×渠道透视表 提交


In [ ]:
import pandas as pd

orders = pd.DataFrame(
    {
        "region": ["华东", "华东", "华南", "华南", "华北"],
        "channel": ["广告", "自然", "广告", "自然", "广告"],
        "amount": [620, 410, 530, 380, 470],
    }
)
summary = (
    orders.groupby(["region", "channel"])
    .agg(
        sales=("amount", "sum"),
        order_count=("amount", "size"),
        average_order=("amount", "mean"),
    )
    .reset_index()
)
pivot = summary.pivot(
    index="region", columns="channel", values="sales"
).fillna(0)
print(summary)
print(pivot)


## 29.14 小结

使用groupby、agg、transform、pivot_table和crosstab回答分组问题。

**迁移思考**：

1. 如果需要计算每个地区销售额占全国的比例，应该用 agg 还是 transform？为什么？
2. 为什么透视表中的缺失组合不能无条件填0？什么情况下填0是合理的？



### 29.14.1 你已经掌握

- 执行分组聚合
- 一次计算多个指标
- 保留原行的组内计算
- 构建透视表和交叉表



### 29.14.2 验收标准

- 输入、计算和输出单元格完整。
- 关键变量类型、形状或数值可核对。
- 结论引用输出证据，并注明适用范围。



### 29.14.3 需要注意

- 使用mean却把结果描述为合计
- groupby后忘记处理索引
- 把缺失组合无条件填0



### 29.14.4 完成检查

- [ ] 能够执行分组聚合
- [ ] 能够一次计算多个指标
- [ ] 能够保留原行的组内计算
- [ ] 能够构建透视表和交叉表



### 29.14.5 排错顺序

1. 从上到下重新运行依赖单元格。
2. 检查变量类型、列名、形状和缺失值。
3. 缩小输入范围，定位产生错误的最小步骤。
4. 修复后重新运行完整流程。

